---
title: "5.2 Folders, paths, notebooks and scripts"
---

::: {.lesson-tools}
[Open in Colab](https://colab.research.google.com/github/ningzhanglab/Python-for-Beginners/blob/main/lessons/05-workflow/02-folders-paths-scripts.ipynb){.btn .btn-primary}
[Download notebook](../../downloads/05-workflow/02-folders-paths-scripts.ipynb){.btn .btn-outline-secondary}
:::

**Run in your browser.** Colab needs a Google account to run and save work. If the GitHub link cannot open, download this notebook and use **File → Upload notebook** at [Colab](https://colab.research.google.com/). You can read the website without an account. Each notebook starts afresh.

**Time:** about 35–40 minutes, including practice. **Before you begin:** Read [5.1 Optional local installation](01-local-installation.ipynb), or stay in the browser and run the prepared example here.

## You will learn

- Separate unchanged inputs, analysis code and generated results.
- Run a portable script without relying on the current working folder.
- Package a small analysis with source, versions, results notes and rerun instructions.

## Why it matters

A notebook is useful for exploration and explanation. A **script** is a text file of Python commands that can run from top to bottom in a new process. Reproducibility means another person can follow documented steps from the inputs to the results. A rerun does not prove the question, measurements or causal interpretation are sound. We start with made-up filenames and then turn the running country question into a small shareable folder.

## Worked example

### Prepare the analysis libraries

Run this prepared cell before importing analysis packages. It uses the course dependency pins and installs only if required. If it asks for a restart, restart the notebook session and run from the top. Installation needs internet; no paid software is needed.

In [ ]:
#| code-fold: true
#| code-summary: "Prepared analysis-library setup (run first)"
# Generated from requirements.txt; installs only in this Python runtime.
import importlib.metadata
from pathlib import Path
import subprocess
import sys

requested = ['statsmodels==0.15.0']
needs_install = False
for requirement in requested:
    name, version = requirement.split("==")
    try:
        needs_install = needs_install or importlib.metadata.version(name) != version
    except importlib.metadata.PackageNotFoundError:
        needs_install = True
if needs_install:
    loaded = any(name in sys.modules for name in ["numpy", "pandas", "scipy", "statsmodels", "linearmodels"])
    constraints = Path("scratch/course-analysis-constraints.txt")
    constraints.parent.mkdir(parents=True, exist_ok=True)
    constraints.write_text('# Python 3.12; tested on macOS. CI also installs this file on Ubuntu.\n# Direct dependencies and their resolved transitive dependencies are pinned.\n# Add a library only when an available lesson or a course check needs it.\nappnope==1.0.0; sys_platform == "darwin"\nasttokens==3.0.2\nattrs==26.1.0\ncomm==0.2.3\ncontourpy==1.4.0\ncycler==0.12.1\ndebugpy==1.8.22\net_xmlfile==2.0.0\nexecuting==2.3.0\nfastjsonschema==2.22.3\nfonttools==4.68.0\nipykernel==7.4.0\nipython==9.18.1\nipython_pygments_lexers==1.1.1\njedi==0.20.1\njsonschema==4.26.0\njsonschema-specifications==2025.9.1\njupyter_client==8.10.0\njupyter_core==5.9.1\nkiwisolver==1.5.1\nmatplotlib==3.11.2\nmatplotlib-inline==0.2.2\nnbclient==0.11.0\nnbformat==5.11.1\nnest-asyncio2==1.7.4\nnumpy==2.5.4\nopenpyxl==3.1.5\npackaging==26.3\npandas==3.1.1\nparso==0.8.7\npexpect==4.9.0; sys_platform != "win32"\npillow==13.0.0\nplatformdirs==4.14.5\nprompt_toolkit==3.0.53\npsutil==7.2.2\nptyprocess==0.7.0; sys_platform != "win32"\npure_eval==0.2.4\nPygments==2.21.0\npyparsing==3.3.3\npython-dateutil==4.0.0\nPyYAML==6.0.3\npyzmq==27.3.0\nreferencing==0.37.0\nrpds-py==2026.9.1\nstack-data==0.6.3\ntornado==6.6.1\ntraitlets==5.16.1\ntyping_extensions==4.16.0\ntzdata==2026.6\nwcwidth==0.9.4\n\n# Statistics and fixed-effects models, with their newly required dependencies.\nformulaic==1.2.2\ninterface_meta==2.0.1\nlinearmodels==7.1\nmypy_extensions==1.2.0\nnarwhals==2.28.0\npatsy==1.0.3\npyhdfe==0.2.1\nscipy==1.18.1\nstatsmodels==0.15.0\nwrapt==2.5.2\n', encoding='utf-8')
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--constraint", str(constraints), *requested], check=True)
    if loaded:
        raise RuntimeError("Packages installed. Restart the notebook session, then run all cells from the top so updated libraries are used together.")
print("Analysis libraries ready:", ", ".join(requested))

Look for **Analysis libraries ready**. These packages run inside the notebook session; this does not install Python on your laptop.

### Build paths instead of typing computer-specific separators

A relative path starts from a working folder. Path joins choose the correct separators for the operating system. These paths and made-up numbers describe a fictional Northland input; the dictionary is only an in-memory outline, and no survey file is created.

In [ ]:
from pathlib import Path
fictional_root = Path("northland-analysis")
fictional_input = fictional_root / "data" / "raw" / "survey.csv"
fictional_output = fictional_root / "results" / "table.csv"
print(fictional_input)
print(fictional_output)
fictional_measurements = {"year": [2000, 2020], "income": [100.0, 150.0]}
print("Fictional input outline:", fictional_measurements)

The input and output are separate. A relative path is portable only when the starting folder is defined. A script can instead anchor its default paths to Path(__file__).resolve().parent; __file__ is the script’s own location, and normally does not exist in a notebook cell.

### Prepare the small project

This generated cell writes the course’s example scripts, README and a copy of its single dependency file into scratch/income-life-project. It copies the committed raw TSV locally or downloads its checksum-verified public source in a fresh browser session. The original course files remain inputs.

In [ ]:
#| code-fold: true
#| code-summary: "Prepared portable-project files"
from pathlib import Path
import subprocess
import sys

project_dir = Path('scratch/income-life-project').resolve()
project_dir.mkdir(parents=True, exist_ok=True)
project_sources = {'analyze.py': '"""Run the descriptive income/life-expectancy analysis from a verified raw file."""\nimport argparse\nfrom pathlib import Path\nimport platform\n\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nimport statsmodels\nimport statsmodels.formula.api as smf\n\nfrom fetch_data import ensure_data\n\nROOT = Path(__file__).resolve().parent\n\n\ndef analyze(year=2007):\n    source = ensure_data()\n    countries = pd.read_csv(source, sep=\'\\t\').rename(columns={\n        \'continent\': \'region\', \'lifeExp\': \'life_exp\', \'pop\': \'population\', \'gdpPercap\': \'gdp_pc\'})\n    if countries.duplicated([\'country\', \'year\']).any():\n        raise ValueError(\'Duplicate country-year keys need review.\')\n    sample = countries.loc[countries[\'year\'] == year, [\'country\', \'life_exp\', \'gdp_pc\']].dropna().copy()\n    if len(sample) < 3 or not sample[\'gdp_pc\'].gt(0).all():\n        raise ValueError(\'Select an observed year with at least three complete, positive-income observations.\')\n    sample[\'log_gdp_pc\'] = np.log(sample[\'gdp_pc\'])\n    model = smf.ols(\'life_exp ~ log_gdp_pc\', sample, missing=\'raise\').fit(cov_type=\'HC1\', use_t=True)\n    interval = model.conf_int().loc[\'log_gdp_pc\']\n    results = ROOT / \'results\'\n    results.mkdir(exist_ok=True)\n    table = pd.DataFrame([{\n        \'year\': year, \'n_countries\': int(model.nobs), \'term\': \'log_gdp_pc\',\n        \'estimate_years_per_log_unit\': model.params[\'log_gdp_pc\'],\n        \'HC1_standard_error\': model.bse[\'log_gdp_pc\'], \'ci95_lower\': interval.iloc[0],\n        \'ci95_upper\': interval.iloc[1], \'covariance\': \'HC1; t-based approximate intervals\',\n    }])\n    table.to_csv(results / f\'model-{year}.csv\', index=False, float_format=\'%.10f\')\n    plt.rcParams.update({\'font.size\': 12})\n    figure, axis = plt.subplots(figsize=(6, 4), layout=\'constrained\', facecolor=\'white\')\n    axis.scatter(sample[\'gdp_pc\'], sample[\'life_exp\'], color=\'#21645a\', alpha=0.8, edgecolor=\'white\', linewidth=0.4)\n    axis.set_xscale(\'log\')\n    axis.set(xlabel=\'GDP per person (inflation-adjusted US dollars; log axis)\',\n             ylabel=\'Life expectancy at birth (years)\', title=f\'Income and life expectancy · {year}\')\n    axis.grid(axis=\'y\', alpha=0.25)\n    figure.savefig(results / f\'income-life-{year}.png\', dpi=300, facecolor=\'white\')\n    plt.close(figure)\n    notes = (f\'Historical Gapminder teaching extract via Jennifer Bryan (CC0). Year: {year}. \'\n             f\'N={len(sample)} complete country observations; equal country weights. \'\n             \'Outcome: life expectancy at birth in years. Predictor: natural log of GDP per person \'\n             "in the extract\'s inflation-adjusted US dollars; base year unspecified. "\n             \'HC1 standard errors; t-based approximate 95% intervals. Association, not a causal estimate. \'\n             \'Alt text: Scatter of country GDP per person on a log axis and life expectancy, \'\n             \'with a broad positive association and substantial dispersion. \'\n             f\'Environment: Python {platform.python_version()}, pandas {pd.__version__}, \'\n             f\'statsmodels {statsmodels.__version__}.\\n\')\n    (results / f\'notes-{year}.txt\').write_text(notes, encoding=\'utf-8\')\n    print(table.to_string(index=False))\n    print(\'Results:\', results)\n    return table\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--year\', type=int, default=2007, help=\'One of the observed five-year points, 1952–2007.\')\n    args = parser.parse_args()\n    analyze(args.year)\n', 'fetch_data.py': '"""Fetch or verify the exact public, CC0 teaching source; no credentials needed."""\nimport argparse\nimport hashlib\nfrom pathlib import Path\nfrom urllib.request import urlopen\n\nSOURCE = \'https://raw.githubusercontent.com/jennybc/gapminder/391b5a40d3574c3b89d889dd7d72f7f3fdb4a70c/inst/extdata/gapminder.tsv\'\nSHA256 = \'f74275b4a2f4a7c8e3a09a752f9d6e43d1759d78d5cbfdebaaa3e1a6efc68f78\'\nDATA = Path(__file__).resolve().parent / \'data/raw/gapminder.tsv\'\n\n\ndef ensure_data(destination=DATA, source=None):\n    destination = Path(destination)\n    if source is not None:\n        raw = Path(source).read_bytes()\n    elif destination.exists():\n        raw = destination.read_bytes()\n    else:\n        with urlopen(SOURCE, timeout=30) as response:\n            raw = response.read()\n    if hashlib.sha256(raw).hexdigest() != SHA256:\n        raise ValueError(\'Data checksum changed. Review the source; do not silently accept a replacement.\')\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    if not destination.exists() or source is not None:\n        temporary = destination.with_suffix(\'.tmp\')\n        temporary.write_bytes(raw)\n        temporary.replace(destination)\n    return destination\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--from-file\', type=Path, help=\'Use an existing exact source for an offline run.\')\n    args = parser.parse_args()\n    print(\'Verified data:\', ensure_data(source=args.from_file))\n', 'README.md': "# A small analysis someone else can rerun\n\nQuestion: do richer countries have longer life expectancy in this historical teaching extract? This is a descriptive association, not a causal design.\n\nThis folder's downloadable version contains the original `data/raw/gapminder.tsv`, its two Python scripts and a generated copy of the course's sole `requirements.txt`. The raw TSV is never edited. Outputs go to `results/`, beside the scripts, even when you launch the script from a different working folder.\n\nFrom this extracted folder, use Python 3.12. If you have uv installed:\n\n```sh\nuv venv --python 3.12\nuv pip sync requirements.txt\n```\n\nOn macOS/Linux run:\n\n```sh\n.venv/bin/python analyze.py\n```\n\nOn Windows PowerShell run:\n\n```powershell\n.venv\\Scripts\\python analyze.py\n```\n\nThe script produces a 2007 scatter, a selected-coefficient CSV and notes with source, units, sample, uncertainty settings, alt text and environment versions. `--year 2002` writes separate filenames. Both example years contain 142 country observations; the 2007 log-income coefficient is about 7.202802 years per natural-log unit. Small numeric tolerances are appropriate across platforms; font rendering and image bytes need not match across operating systems.\n\n`fetch_data.py` verifies the existing raw file's SHA-256 before use. If it is absent, it downloads the pinned public source. Run the file directly to download/verify it. `--from-file /path/to/gapminder.tsv` supplies an existing exact copy offline. It stops on changed bytes instead of adopting a newer dataset. Creating the Python environment needs internet; with packages and the included raw file present, analysis works offline.\n\n## Source and rights\n\nGapminder historical teaching extract packaged by Jennifer Bryan, commit `391b5a40d3574c3b89d889dd7d72f7f3fdb4a70c`, retrieved 2026-11-11. Original TSV: https://raw.githubusercontent.com/jennybc/gapminder/391b5a40d3574c3b89d889dd7d72f7f3fdb4a70c/inst/extdata/gapminder.tsv . SHA-256: `f74275b4a2f4a7c8e3a09a752f9d6e43d1759d78d5cbfdebaaa3e1a6efc68f78`.\n\nSource data are CC0; see https://creativecommons.org/publicdomain/zero/1.0/ and the upstream package DESCRIPTION. All 1704 source observations are retained; analysis selects one year and renames columns in memory. The course's original code, text and figures have no selected license. No open license is implied for those materials.\n\nBefore sharing research, document your own question, source rights, sample decisions, versions, limitations and the exact command. A successful rerun verifies a computation; it does not establish that the question or model is appropriate.\n", 'requirements.txt': '# Python 3.12; tested on macOS. CI also installs this file on Ubuntu.\n# Direct dependencies and their resolved transitive dependencies are pinned.\n# Add a library only when an available lesson or a course check needs it.\nappnope==1.0.0; sys_platform == "darwin"\nasttokens==3.0.2\nattrs==26.1.0\ncomm==0.2.3\ncontourpy==1.4.0\ncycler==0.12.1\ndebugpy==1.8.22\net_xmlfile==2.0.0\nexecuting==2.3.0\nfastjsonschema==2.22.3\nfonttools==4.68.0\nipykernel==7.4.0\nipython==9.18.1\nipython_pygments_lexers==1.1.1\njedi==0.20.1\njsonschema==4.26.0\njsonschema-specifications==2025.9.1\njupyter_client==8.10.0\njupyter_core==5.9.1\nkiwisolver==1.5.1\nmatplotlib==3.11.2\nmatplotlib-inline==0.2.2\nnbclient==0.11.0\nnbformat==5.11.1\nnest-asyncio2==1.7.4\nnumpy==2.5.4\nopenpyxl==3.1.5\npackaging==26.3\npandas==3.1.1\nparso==0.8.7\npexpect==4.9.0; sys_platform != "win32"\npillow==13.0.0\nplatformdirs==4.14.5\nprompt_toolkit==3.0.53\npsutil==7.2.2\nptyprocess==0.7.0; sys_platform != "win32"\npure_eval==0.2.4\nPygments==2.21.0\npyparsing==3.3.3\npython-dateutil==4.0.0\nPyYAML==6.0.3\npyzmq==27.3.0\nreferencing==0.37.0\nrpds-py==2026.9.1\nstack-data==0.6.3\ntornado==6.6.1\ntraitlets==5.16.1\ntyping_extensions==4.16.0\ntzdata==2026.6\nwcwidth==0.9.4\n\n# Statistics and fixed-effects models, with their newly required dependencies.\nformulaic==1.2.2\ninterface_meta==2.0.1\nlinearmodels==7.1\nmypy_extensions==1.2.0\nnarwhals==2.28.0\npatsy==1.0.3\npyhdfe==0.2.1\nscipy==1.18.1\nstatsmodels==0.15.0\nwrapt==2.5.2\n'}
for name, source in project_sources.items():
    (project_dir / name).write_text(source, encoding='utf-8')
command = [sys.executable, str(project_dir / 'fetch_data.py')]
local_raw = Path('data/raw/gapminder.tsv')
if local_raw.exists():
    command += ['--from-file', str(local_raw.resolve())]
subprocess.run(command, check=True)
print('Project files ready:', project_dir)

Look for **Project files ready**. The folder contains analyze.py, fetch_data.py, README.md, requirements.txt and data/raw/gapminder.tsv. The raw file stays unchanged. Generated tables and figures go to results/. This prepared cell is folded because the scripts are provided for study, not for retyping.

### Read the roles before running

`fetch_data.py` obtains or verifies the pinned original TSV. `analyze.py` checks unique country-year keys, selects complete observations for one year, creates log income, fits the same HC1 OLS model as lesson 4.2 and writes a chart, CSV and notes. The full coefficient is read from the result object. The README states a rerun command, source rights, units, sample and limitations.

The script locates inputs relative to **its own file**, so a person does not need your username or folder names. The environment's Python is explicit. Inside a notebook, `sys.executable` identifies that interpreter; `subprocess.run` starts a separate process. `check=True` makes a failed child command stop rather than pretending results were produced. This prepared orchestration is provided; the essential idea is a fresh process with explicit inputs.

A notebook's order on the screen is not necessarily the order in which cells were previously run. Use **Restart and run all** before sharing a notebook. A script avoids hidden cell order but can still depend on undocumented files, environment variables or packages unless you specify them.

### Run from an unrelated working folder

Launch the script in a temporary folder using its absolute script path. A temporary folder exists only for the operation. Its working folder deliberately contains none of the input files.

In [ ]:
import tempfile
with tempfile.TemporaryDirectory(prefix="course-rerun-") as temporary:
    completed = subprocess.run([sys.executable, str(project_dir / "analyze.py")], cwd=temporary, check=True)
print("Exit status:", completed.returncode)

Exit status 0 means this command finished successfully. The script still found its own raw input and wrote into the project’s results folder. In the 2007 cross-section expect 142 countries and the log-income coefficient near 7.202802 years per log unit. Equal country weights and HC1 uncertainty do not establish causation.

### Inspect the generated table and notes

Read the output back instead of trusting a success message. Keep the table and its measurement/source notes together.

In [ ]:
import pandas as pd
portable_result = pd.read_csv(project_dir / "results/model-2007.csv")
print(portable_result[["year", "n_countries", "estimate_years_per_log_unit"]])
print((project_dir / "results/notes-2007.txt").read_text())

The CSV contains numeric results, while the notes explain units, sample, uncertainty, provenance and environment versions. Neither a filename nor a coefficient alone communicates the meaning of an analysis.

### View the chart produced by the separate process

Display the saved image; this is the output of the script, not a second hand-made chart.

In [ ]:
#| label: fig-portable-result
#| fig-alt: "2007 country scatter from the portable project: a broad positive association between log-scaled GDP per person and life expectancy, with substantial dispersion."
#| fig-cap: "Gapminder teaching extract via Jennifer Bryan, CC0. Equal country weights; descriptive association."
from IPython.display import Image
Image(filename=str(project_dir / "results/income-life-2007.png"))

The axes retain the measurement units and observation year. A 300-DPI export is useful for a paper, but source provenance and interpretation still belong in the accompanying notes.

### Package the inputs and instructions

Create a ZIP containing only the five required source/input files. The environment, caches and generated outputs are intentionally excluded. Also copy this run’s chart, table and notes into the website’s download area.

In [ ]:
import zipfile
output_dir = Path("downloads/outputs")
output_dir.mkdir(parents=True, exist_ok=True)
archive_file = output_dir / "income-life-project.zip"
project_members = ["analyze.py", "fetch_data.py", "README.md", "requirements.txt", "data/raw/gapminder.tsv"]
with zipfile.ZipFile(archive_file, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for name in sorted(project_members):
        entry = zipfile.ZipInfo("income-life-project/" + name, date_time=(2026, 11, 11, 0, 0, 0))
        entry.compress_type = zipfile.ZIP_DEFLATED
        archive.writestr(entry, (project_dir / name).read_bytes())
for source_name, exported_name in [("income-life-2007.png", "workflow-income-life-2007.png"), ("model-2007.csv", "workflow-model-2007.csv"), ("notes-2007.txt", "workflow-notes-2007.txt")]:
    (output_dir / exported_name).write_bytes((project_dir / "results" / source_name).read_bytes())
print("Portable archive:", archive_file)
print("Included source files:", len(project_members))

Download the [portable project ZIP](../../downloads/outputs/income-life-project.zip), [figure](../../downloads/outputs/workflow-income-life-2007.png), [table](../../downloads/outputs/workflow-model-2007.csv) and [notes](../../downloads/outputs/workflow-notes-2007.txt). In Colab use the Files panel under downloads → outputs. Extract the ZIP, create the Python 3.12 environment from requirements.txt and run analyze.py as the README describes. The included raw file makes analysis offline once packages are installed. Do not share a whole environment folder or private inputs merely because they are in your working directory.

## Try it

Work in your own copy. Add a cell for each task and attempt it before reading a solution.

### 1. Rerun 2002 without overwriting the 2007 results.

<details><summary>Solution 1</summary>

Run this solution after your own attempt.

In [ ]:
subprocess.run([sys.executable, str(project_dir / "analyze.py"), "--year", "2002"], check=True)
result_2002 = pd.read_csv(project_dir / "results/model-2002.csv")
print(result_2002[["year", "n_countries", "estimate_years_per_log_unit"]])

The year argument writes separate output filenames. This is still a cross-country comparison for one observed year, not a panel fixed-effects model or a causal time effect.

</details>

### 2. Check that the shared archive contains only the intended files.

<details><summary>Solution 2</summary>

Run this solution after your own attempt.

In [ ]:
with zipfile.ZipFile(archive_file) as archive:
    archived_names = archive.namelist()
print("\n".join(archived_names))

There are five entries, with no .venv or cache files. A smaller package is easier to inspect, and its README tells the next person which files are inputs and which results will be regenerated.

</details>

### 3. Name the facts a reader needs alongside the CSV.

<details><summary>Solution 3</summary>

Run this solution after your own attempt.

In [ ]:
notes = (project_dir / "results/notes-2007.txt").read_text()
for phrase in ["N=142", "HC1", "CC0", "Association"]:
    print(phrase, phrase in notes)

These checks locate notes; they do not independently audit the analysis. Read the complete notes and explain sample, units, source and uncertainty in your own words.

</details>

## When it goes wrong

These demonstrations catch a specific intended error so the notebook continues. Unexpected errors elsewhere should still stop execution.

### A relative path is interpreted from the wrong folder

This intentionally reads a missing file rather than the project input.

In [ ]:
try:
    Path("scratch/not-this-project/data.tsv").read_text()
except FileNotFoundError as error:
    print(type(error).__name__, "Check the starting folder and the documented project paths.")

Changing the working folder at random is fragile. A reusable script should have explicit input paths or stable paths relative to its own location.

### An unobserved year has no analysis sample

Capture the script’s expected failed process for 2010, which is beyond this historical extract.

In [ ]:
failed = subprocess.run([sys.executable, str(project_dir / "analyze.py"), "--year", "2010"], capture_output=True, text=True)
print("Rejected unobserved year:", failed.returncode != 0)
print("Sample explanation present:", "observed year" in failed.stderr)

A failed sample check is information to inspect, not a reason to replace missing observations with invented values. The previously valid output files are not evidence that this failed run succeeded.

## Recap

- Keep raw inputs separate from generated results.
- A script runs afresh, but its paths and dependencies still need documentation.
- Check a rerun’s output, not only its exit status.
- Package code, small permitted inputs and instructions; retain rights and interpretation notes.

**Next:** [5.3 Public data by code](03-public-data.ipynb).